# Week 7 workshop · Particle swarm optimisation

<span class="workshop-download-enabled" aria-hidden="true"></span>


# Context


Particle swarm optimisation uses personal and globally shared records to guide a population through a solution space. Finding a good solution does not necessarily show that this shared search was useful: the same evaluation budget might succeed through random sampling alone. We therefore compare PSO with independent random search, then remove shared information to isolate what it contributes. These baselines determine which claims the model output can support. Because the searches are stochastic, each comparison uses the same evaluation budget and repeated seeds.

# Specify the PSO investigation

## PSO baseline

For a minimisation problem, particle $i$ stores a candidate $\mathbf{x}_i^n$, a velocity $\mathbf{v}_i^n$ and its personal-best position $\mathbf{p}_i$. Every particle can use the best-so-far position $\mathbf{g}$ shared globally.

At iteration $n$,

$$
\mathbf v_i^{n+1}=w\mathbf v_i^n
+c_1\mathbf r_{1,i}^n\odot[\mathbf p_i-\mathbf x_i^n]
+c_2\mathbf r_{2,i}^n\odot[\mathbf g-\mathbf x_i^n],
$$

followed by $\mathbf{x}_i^{n+1}=\mathbf{x}_i^n+\mathbf{v}_i^{n+1}$. Here $\odot$ denotes componentwise multiplication. The coefficient $w$ is inertia, while $c_1$ and $c_2$ weight personal and shared information. Every component of $\mathbf r_{1,i}^n$ and $\mathbf r_{2,i}^n$ is an independent uniform draw from $[0,1]$.

Particles begin at random positions within the feasible bounds and with small random velocities. In each coordinate, the initial velocity is no more than 10% of that coordinate's range in either direction. This initial movement matters when $c_2=0$: personal best initially equals current position, so without a non-zero velocity the particle would not move. Positions that leave the feasible region are placed at the nearest boundary.


# Establish the experimental baseline

The following two cells provide the PSO implementation. This week, the baseline checks are also supplied. Run them before the investigation and use their structure to see how a claim about code is turned into a small, controlled test. In Week 8, you will construct this kind of check for reused code yourself.

In [8]:
import numpy as np
import matplotlib.pyplot as plt

def objective_landscape(x):
    x = np.asarray(x, dtype=float)
    dx = x[..., 0] - 1.3
    dy = x[..., 1] + 0.8
    return (
        0.035 * (dx**2 + 1.15 * dy**2)
        + (np.sin(1.15 * dx) + 0.42 * np.sin(2.05 * dy))**2
        + 0.65 * (np.sin(0.78 * dy)
                  + 0.38 * np.sin(2.35 * dx + 0.2 * dy))**2
    )


In [9]:
def initialise_swarm(n_particles, n_dims, bounds, objective, rng,
                     initial_velocity_fraction=0.10):
    if not isinstance(n_particles, (int, np.integer)) or n_particles < 1:
        raise ValueError("n_particles must be a positive integer")
    if not isinstance(n_dims, (int, np.integer)) or n_dims < 1:
        raise ValueError("n_dims must be a positive integer")
    if initial_velocity_fraction < 0:
        raise ValueError("initial_velocity_fraction must be non-negative")

    lower = np.broadcast_to(np.asarray(bounds[0], dtype=float), (n_dims,))
    upper = np.broadcast_to(np.asarray(bounds[1], dtype=float), (n_dims,))
    if not np.all(np.isfinite(lower)) or not np.all(np.isfinite(upper)):
        raise ValueError("bounds must be finite")
    if np.any(upper <= lower):
        raise ValueError("every upper bound must exceed its lower bound")

    positions = rng.uniform(lower, upper, size=(n_particles, n_dims))
    velocity_limit = initial_velocity_fraction * (upper - lower)
    velocities = rng.uniform(-velocity_limit, velocity_limit,
                             size=(n_particles, n_dims))
    personal_best = positions.copy()
    personal_scores = np.asarray(objective(personal_best), dtype=float)
    if personal_scores.shape != (n_particles,):
        raise ValueError("objective must return one score per particle")
    if not np.all(np.isfinite(personal_scores)):
        raise ValueError("objective returned a non-finite score")
    return positions, velocities, personal_best, personal_scores


def pso_step(positions, velocities, personal_best, personal_scores,
             objective, bounds, rng, inertia=0.72,
             personal_weight=1.49, shared_weight=1.49, max_speed=None):
    if max_speed is not None and max_speed <= 0:
        raise ValueError("max_speed must be positive when supplied")

    global_best = personal_best[np.argmin(personal_scores)].copy()
    r_personal = rng.random(positions.shape)
    r_shared = rng.random(positions.shape)
    velocities = (
        inertia * velocities
        + personal_weight * r_personal * (personal_best - positions)
        + shared_weight * r_shared * (global_best - positions)
    )
    if max_speed is not None:
        speed = np.linalg.norm(velocities, axis=1, keepdims=True)
        velocities *= np.minimum(1.0, max_speed / np.maximum(speed, 1e-12))

    # Boundary rule: clip the position and retain the calculated velocity.
    positions = np.clip(positions + velocities, bounds[0], bounds[1])
    scores = np.asarray(objective(positions), dtype=float)
    if scores.shape != personal_scores.shape:
        raise ValueError("objective must return one score per particle")
    if not np.all(np.isfinite(scores)):
        raise ValueError("objective returned a non-finite score")

    personal_best = personal_best.copy()
    personal_scores = personal_scores.copy()
    improved = scores < personal_scores
    personal_best[improved] = positions[improved]
    personal_scores[improved] = scores[improved]
    return positions, velocities, personal_best, personal_scores, scores


def run_pso(objective=objective_landscape, seed=7, n_particles=30, n_dims=2,
            bounds=((-6.0, -5.0), (6.0, 5.0)), steps=100,
            initial_velocity_fraction=0.10, **update_parameters):
    if not isinstance(steps, (int, np.integer)) or steps < 0:
        raise ValueError("steps must be a non-negative integer")

    rng = np.random.default_rng(seed)
    initial = initialise_swarm(
        n_particles, n_dims, bounds, objective, rng,
        initial_velocity_fraction=initial_velocity_fraction,
    )
    positions, velocities, personal_best, personal_scores = initial

    position_history = [positions.copy()]
    best_scores = [personal_scores.min()]
    diversity = [np.mean(np.linalg.norm(
        positions - positions.mean(axis=0), axis=1
    ))]

    for _ in range(steps):
        positions, velocities, personal_best, personal_scores, _ = pso_step(
            positions, velocities, personal_best, personal_scores,
            objective, bounds, rng, **update_parameters,
        )
        position_history.append(positions.copy())
        best_scores.append(personal_scores.min())
        diversity.append(np.mean(np.linalg.norm(
            positions - positions.mean(axis=0), axis=1
        )))

    best_index = int(np.argmin(personal_scores))
    return {
        "positions": np.asarray(position_history),
        "best_scores": np.asarray(best_scores),
        "diversity": np.asarray(diversity),
        "personal_best": personal_best.copy(),
        "personal_scores": personal_scores.copy(),
        "best_position": personal_best[best_index].copy(),
        "best_score": float(personal_scores[best_index]),
        "objective_evaluations": n_particles * (steps + 1),
    }


## Check the supplied implementation

Before comparing algorithms, check that the supplied PSO behaves as it should. Each check translates a modelling claim into a small controlled example and an assertion about its output. This is an application of the ladder of abstraction. The checks are supplied this week; in Week 8 you will design them yourself.

The next cell checks that:

1. the objective has its known minimum value at $(1.3,-0.8)$;
2. a fixed seed reproduces the same run;
3. every recorded particle position remains inside the feasible region;
4. a newly visited better position replaces the relevant personal-best record; and
5. a run with $N$ particles and $K$ updates makes $N(K+1)$ objective evaluations.

Run the cell and read each example alongside its assertion. If a check fails, do not continue to the comparison.


In [10]:
# 1. The stated optimum has objective value zero.
known_minimum = np.array([1.3, -0.8])
assert np.isclose(objective_landscape(known_minimum), 0.0)

# 2. A fixed seed reproduces the same trajectory and measurements.
run_a = run_pso(seed=19, steps=40)
run_b = run_pso(seed=19, steps=40)
for key in ("positions", "best_scores", "diversity"):
    assert np.array_equal(run_a[key], run_b[key])

# 3. Every recorded position remains within the feasible bounds.
lower = np.array([-6.0, -5.0])
upper = np.array([6.0, 5.0])
assert np.all(run_a["positions"] >= lower)
assert np.all(run_a["positions"] <= upper)

# 4. A better visit replaces personal memory; a worse visit does not.
def sphere(x):
    return np.sum(np.asarray(x, dtype=float) ** 2, axis=-1)

positions = np.array([[2.0, 0.0], [0.0, 0.0]])
velocities = np.array([[-1.0, 0.0], [1.0, 0.0]])
personal_best = positions.copy()
personal_scores = sphere(personal_best)
_, _, new_best, new_scores, visited_scores = pso_step(
    positions, velocities, personal_best, personal_scores,
    objective=sphere, bounds=((-3.0, -3.0), (3.0, 3.0)),
    rng=np.random.default_rng(5), inertia=1.0,
    personal_weight=0.0, shared_weight=0.0,
)
assert visited_scores[0] < personal_scores[0]
assert np.array_equal(new_best[0], np.array([1.0, 0.0]))
assert visited_scores[1] > personal_scores[1]
assert np.array_equal(new_best[1], np.array([0.0, 0.0]))

# 5. Initial evaluation plus K updates evaluates every particle K+1 times.
N, K = 17, 23
counted_run = run_pso(n_particles=N, steps=K, seed=8)
assert counted_run["objective_evaluations"] == N * (K + 1)

print("All supplied baseline checks passed.")


All supplied baseline checks passed.


# Investigate


## Design the baseline comparison

Compare three conditions:

1. **independent random search:** a null model that samples the solution space without memory or sharing;
2. **PSO without shared information:** set `shared_weight=0` while retaining personal memory and motion;
3. **PSO with shared information:** use the stated PSO settings.

The first comparison asks whether structured search improves on chance. The second isolates the contribution of globally shared information. Use the same objective-evaluation budget and repeated seeds in every condition.

Assess:

- **reliability:** the fraction of runs reaching a success criterion chosen in advance;
- **solution quality:** the distribution of final best objective values; and
- **search concentration:** the final diversity of the evaluated positions.

$$
D(n)=\frac{1}{N}\sum_{i=1}^{N}\lVert\mathbf{x}_i^n-\bar{\mathbf{x}}^n\rVert.
$$

| Decision | Your choice and justification |
|---|---|
| prediction for each comparison |  |
| fixed PSO settings |  |
| success criterion |  |
| evaluation budget |  |
| number of repeated seeds |  |
| outputs and summaries |  |

After establishing these baselines, optionally vary `shared_weight` to test sensitivity.


### Supplied comparison helpers

The random-search function evaluates the same number of candidate positions as PSO. The ensemble helpers repeat each condition across a declared collection of seeds.


In [11]:
def run_random_search(objective=objective_landscape, seed=7, n_particles=30,
                      n_dims=2, bounds=((-6.0, -5.0), (6.0, 5.0)),
                      steps=100, **unused_settings):
    rng = np.random.default_rng(seed)
    lower = np.broadcast_to(np.asarray(bounds[0], dtype=float), (n_dims,))
    upper = np.broadcast_to(np.asarray(bounds[1], dtype=float), (n_dims,))
    best_scores, diversity = [], []
    best = np.inf
    for _ in range(steps + 1):
        positions = rng.uniform(lower, upper, size=(n_particles, n_dims))
        scores = np.asarray(objective(positions), dtype=float)
        best = min(best, float(scores.min()))
        best_scores.append(best)
        diversity.append(np.mean(np.linalg.norm(
            positions - positions.mean(axis=0), axis=1
        )))
    return {
        "best_scores": np.asarray(best_scores),
        "diversity": np.asarray(diversity),
        "best_score": float(best),
        "objective_evaluations": n_particles * (steps + 1),
    }


def _ensemble(runner, seeds, **settings):
    runs = [runner(seed=int(seed), **settings) for seed in seeds]
    return {
        "final_best": np.array([run["best_scores"][-1] for run in runs]),
        "final_diversity": np.array([run["diversity"][-1] for run in runs]),
        "evaluations": np.array([run["objective_evaluations"] for run in runs]),
        "runs": runs,
    }


def run_random_ensemble(seeds, **settings):
    return _ensemble(run_random_search, seeds, **settings)


def run_ensemble(shared_weight, seeds, **fixed_settings):
    return _ensemble(
        run_pso, seeds, shared_weight=shared_weight, **fixed_settings
    )


def summarise_ensemble(ensemble, success_threshold):
    final_best = ensemble["final_best"]
    return {
        "runs": len(final_best),
        "success_fraction": np.mean(final_best <= success_threshold),
        "median_final_best": np.median(final_best),
        "median_final_diversity": np.median(ensemble["final_diversity"]),
        "evaluations_per_run": np.unique(ensemble["evaluations"]),
    }


In [12]:
# Use paired seeds so that each condition is repeated from the same seed labels.
seeds = np.arange(60) + 10_000

# The known minimum is zero; declare success before examining the results.
success_threshold = 1e-3

# Every condition receives 30 x (100 + 1) = 3030 objective evaluations.
fixed_settings = {
    "n_particles": 30,
    "steps": 100,
    "bounds": ((-6.0, -5.0), (6.0, 5.0)),
    "inertia": 0.72,
    "personal_weight": 1.49,
}

random_baseline = run_random_ensemble(seeds, **fixed_settings)
no_sharing = run_ensemble(0.0, seeds, **fixed_settings)
with_sharing = run_ensemble(1.49, seeds, **fixed_settings)

conditions = {
    "independent random search": random_baseline,
    "PSO without sharing": no_sharing,
    "PSO with sharing": with_sharing,
}

for name, ensemble in conditions.items():
    print(name, summarise_ensemble(ensemble, success_threshold))

# Compare success and final solution quality before interpreting diversity.


independent random search {'runs': 60, 'success_fraction': np.float64(0.15), 'median_final_best': np.float64(0.00419778359413237), 'median_final_diversity': np.float64(4.106480931571401), 'evaluations_per_run': array([3030])}
PSO without sharing {'runs': 60, 'success_fraction': np.float64(0.18333333333333332), 'median_final_best': np.float64(0.013350689115082217), 'median_final_diversity': np.float64(3.912195060907883), 'evaluations_per_run': array([3030])}
PSO with sharing {'runs': 60, 'success_fraction': np.float64(1.0), 'median_final_best': np.float64(1.4786339501723442e-11), 'median_final_diversity': np.float64(0.005259161207158436), 'evaluations_per_run': array([3030])}


## Evidence checkpoint

Report the experimental settings, success criterion and common evaluation budget. Compare the success fraction and distribution of final best values for all three conditions. Then use diversity to help explain how the searches differed.

Include one figure that permits a direct comparison. State separately whether structured search improved on random sampling and whether shared information improved on PSO without sharing. Limit both conclusions to the landscape and settings tested.


# Optional transfer


## Design a PSO problem

Choose a problem that could be represented as a search through a continuous solution space. You do not need to find or clean a dataset. Specify:

1. the candidate solution and its feasible bounds;
2. the objective function;
3. what would count as a good solution;
4. whether the true global optimum is known;
5. where local optima might arise;
6. the measurements you would use to assess PSO; and
7. a stopping condition.

Explain how each modelling choice represents the original problem. The main task is the specification, not an implementation.

### Optional implementation

If the objective can be evaluated directly, implement it below and test it with the supplied PSO implementation. Check the returned solution independently where possible and use repeated runs if the search is stochastic.

### Optional extension · a changing landscape

If the objective changes during a run, a final best value no longer describes performance. Design a moving optimum and decide how you would measure:

- **tracking error:** distance or objective difference from the current optimum;
- **accumulated performance:** error across the whole run; and
- **recovery:** how quickly performance returns after a change.

Personal and shared best records may become stale when the landscape changes. State whether they should be retained, discounted or reset, and choose a baseline that faces the same sequence of changes. This is a design question; implementation is optional.


In [ ]:
# Optional: define the objective, bounds and PSO settings for your problem.
# Run repeated searches and check the best result independently where possible.


# Exit



Before leaving, record the strongest conclusion supported by your evidence, its main limitation and the next comparison you would make.
